In [ ]:
import numpy as np
import pandas as pd

In [ ]:
### cell 0 ###

factor = 20
flights_df = pd.read_csv(
    "/home/dias-benchmarks/new_notebooks/nyc-flight/nyc_flights.csv"
)
flights_df = pd.concat([flights_df] * factor, ignore_index=True)

In [ ]:
### cell 1 ###

flights_df.shape, flights_df.columns, flights_df.dtypes

In [ ]:
### cell 2 ###

flights_df.dest.unique()
flights_df.head(10)

In [ ]:
### cell 3 ###

flights_df["dest"][flights_df["dest"] == "SEA"].value_counts()

In [ ]:
### cell 4 ###

flights_df["carrier"][flights_df["dest"] == "SEA"].value_counts()

In [ ]:
### cell 5 ###

flights_df.loc[flights_df["dest"] == "SEA", "tailnum"].nunique()

In [ ]:
### cell 6 ###

flights_df["arr_delay"][flights_df["dest"] == "SEA"].mean()

In [ ]:
### cell 7 ###

# Filter once and compute counts more efficiently
f = flights_df.origin[flights_df.dest.eq("SEA")].value_counts()
# Total is just the sum of those counts (avoids a second filter)
f_total = f.sum()
# Compute proportions
f.loc["EWR"] / f_total, f.loc["JFK"] / f_total

In [ ]:
### cell 8 ###

# Combine both mean calculations in a single groupby to avoid two passes
df = flights_df.groupby(["month", "day"], as_index=False)[
    ["dep_delay", "arr_delay"]
].mean()
# Extract the rows with the maximum mean departure and arrival delays
df.loc[df["dep_delay"].idxmax()], df.loc[df["arr_delay"].idxmax()]

In [ ]:
### cell 9 ###

df

In [ ]:
### cell 10 ###

df = flights_df.groupby(["day", "month"], as_index=False).agg(
    {"arr_delay": np.mean, "dep_delay": np.mean}
)
df["total_delay"] = df["arr_delay"] + df["dep_delay"]
df.sort_values("total_delay", ascending=False).head(1)

In [ ]:
### cell 11 ###

ds = flights_df.dropna(subset=["dep_delay"]).groupby(["month"])["dep_delay"].mean()
ds

In [ ]:
### cell 12 ###

dt = flights_df.dropna(subset=["dep_delay"]).groupby(["hour"])["dep_delay"].mean()
dt

In [ ]:
### cell 13 ###

df = flights_df
# compute speeds via numpy to minimize pandas overhead
speed_arr = df["distance"].values / df["air_time"].values
# assign back to df and filter by max speed using numpy mask
df["speed"] = speed_arr
df[speed_arr == speed_arr.max()]

In [ ]:
### cell 14 ###

count = len(flights_df)
df = flights_df.groupby(["carrier", "flight", "dest"]).size().reset_index(name="Size")
for i in df.index:
    if df.loc[i]["Size"] == 365:
        print(
            "Carrier: %s, Flight: %s, Destination: %s"
            % (df.loc[i]["carrier"], df.loc[i]["flight"], df.loc[i]["dest"])
        )

In [ ]:
### cell 15 ###

gdf = flights_df
counts = gdf.groupby(["carrier", "flight", "dest"]).size().reset_index(name="size")
complete_year = counts[counts["size"] == 365]
complete_year[["carrier", "flight", "dest"]]

In [ ]:
### cell 16 ###

# Filter for June flights, compute mean delays by carrier, and add total_delay
df = (
    flights_df.loc[flights_df["month"] == 6]
    .groupby("carrier", as_index=False)[["arr_delay", "dep_delay"]]
    .mean()
    .assign(total_delay=lambda x: x["arr_delay"] + x["dep_delay"])
)
# Find the carrier with the minimum total_delay and print
min_row = df.loc[df["total_delay"].idxmin()]
print(min_row["carrier"], min_row["total_delay"])
df

In [ ]:
### cell 17 ###

weather_df = pd.read_csv(
    "/home/dias-benchmarks/new_notebooks/nyc-flight/nyc_weather.csv"
)
df = flights_df
df_c = pd.merge(df, weather_df, on=["month", "day", "hour", "origin"])
df_c.head(10)